# Modelforge ASE Calculator: Simple Walkthrough

This notebook demonstrates how to use a potential trained with modelforge as a calculator in the Atomic Simulation Environment (ASE).
It is written for first-time users of both modelforge and ASE.

What you will do in this tutorial:
1. Load a trained NNP model checkpoint.
2. Wrap it with `ModelForgeCalculator`.
3. Build a molecule from a SMILES string.
4. Compute single-point energy and forces.
5. Run geometry optimization and a short molecular dynamics simulation.

In [ ]:
from modelforge.potential.potential import load_inference_model_from_checkpoint

# Helper utilities to load the example model checkpoint bundled with modelforge.
from modelforge.utils.io import get_path_string
from modelforge.ase.tests import data

checkpoint_file_path = f"{get_path_string(data)}/model.ckpt" # This is an example model used in testing 
potential = load_inference_model_from_checkpoint(checkpoint_file_path, jit=False)
print(f"Loaded checkpoint: {checkpoint_file_path}")

To evaluate energies and forces in ASE, create a `ModelForgeCalculator` with the loaded potential and attach it to an ASE `Atoms` object.

In [ ]:
from modelforge.ase.calculator import ModelForgeCalculator

modelforge_calculator = ModelForgeCalculator(potential)


ASE includes tools for building molecules, and modelforge-ase also provides helper functions for a simple SMILES-based workflow.

An ASE `Atoms` object stores the system definition: element identities, 3D positions, and optional simulation metadata (for example cell vectors and periodic boundary settings). By itself, `Atoms` is only a container for structure.

To compute potential energy or forces, you must attach a calculator first (here, `ModelForgeCalculator`). Without a calculator, calls like `atoms.get_potential_energy()` and `atoms.get_forces()` cannot run.

In [ ]:
from modelforge.ase.examples.helper_functions import smiles_to_ase, ase_to_rdkit

# Set optimize=True to run an MMFF94 geometry optimization in RDKit before conversion.
smiles = "NCCCCCCO"
atoms = smiles_to_ase(smiles, optimize=False)

# Attach a calculator before requesting energy/forces from ASE.
atoms.calc = modelforge_calculator

You can view the 3D structure directly in ASE. (A separate RDKit view is also possible using the helper conversion utilities.)

In [ ]:
from ase.visualize import view
view(atoms, viewer='x3d')

In [ ]:
# Optional alternative viewer (install nglview first).
# from nglview import show_ase
# show_ase(atoms)

### Single-point energy and force computation
With the calculator attached, ASE can evaluate potential energy and per-atom forces. `ModelForgeCalculator` handles unit conversion from modelforge internal units to ASE units (eV and angstrom).

In [ ]:
pe = atoms.get_potential_energy()
forces = atoms.get_forces()
print(f"Potential energy: {pe:.6f} eV")
print("Forces (eV/angstrom):")
print(forces)

### Geometry optimization
ASE offers several structure optimizers. Here we use BFGS and stop when the maximum force falls below 0.05 eV/angstrom.

In [ ]:
from ase.optimize import BFGS

opt = BFGS(atoms, logfile=f"{smiles}_opt.log", trajectory=f"{smiles}_opt.traj")
opt.run(fmax=0.05)

## Molecular dynamics simulation
You can also run MD with ASE. This example runs short Langevin dynamics at 298 K and writes trajectory/log files.

In [ ]:
import ase.units as ase_units
from ase.io.trajectory import Trajectory
from ase.md import Langevin
from ase.io import write, read

trajectory_name = f"{smiles}_sim.traj"
write(f"{smiles}_system.pdb", atoms)

traj = Trajectory(trajectory_name, "w", atoms)

dyn = Langevin(
    atoms,
    timestep=1.0 * ase_units.fs,
    temperature_K=298.0,  # Kelvin
    friction=1.0 / ase_units.fs,
    trajectory=traj,
    logfile=f"{smiles}_md.log",
)
dyn.run(100)

# Convert ASE trajectory to XYZ so it can be opened in many viewers.
frames = read(trajectory_name, ":")
write(f"{smiles}_sim.xyz", frames, format="xyz")

In [ ]:
for atom in atoms:
    print(atom.number)
    print(atom.position)